# 🚀 Token-Aware 3D Delta — Dataset Generation

Generates **X{N,L,E,D}** and **Y{N,L,E,1}** tensors from SQuAD for learning Δ(E,L,D).

## 1 · Environment Setup

In [1]:
import os
import sys

miniconda_path = f"{os.environ.get('HOME', '')}/miniconda/bin"
os.environ["PATH"] = f"{miniconda_path}:" + os.environ.get("PATH", "")

# Cache dirs -> scratch (absolute path, NOT ~/scratch1)
SCRATCH = os.environ.get("SCRATCH_DIR", "/scratch1/kelidari")
os.environ["HF_HOME"] = f"{SCRATCH}/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = f"{SCRATCH}/hf_cache"
os.environ["TMPDIR"] = f"{SCRATCH}/tmp"
os.environ['WANDB_API_KEY'] = "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"

os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

print(f"Conda PATH = {miniconda_path}")
print(f"HF_HOME    = {os.environ['HF_HOME']}")
print(f"TMPDIR     = {os.environ['TMPDIR']}")
print("✓ W&B API key configured!")

Conda PATH = /home1/kelidari/miniconda/bin
HF_HOME    = /scratch1/kelidari/hf_cache
TMPDIR     = /scratch1/kelidari/tmp
✓ W&B API key configured!


In [2]:
import os

# Preemptively accept Conda TOS to prevent hanging prompts
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

env_name = "llm_steering"
print("Checking environment status...")
res = os.system(f"conda env list | grep {env_name} > /dev/null")
if res == 0:
    print(f"{env_name} exists! Synchronizing any missing packages ...")
    !conda env update -f environment.yml --prune
else:
    print(f"{env_name} does not exist. Creating fresh environment...")
    !conda env create -f environment.yml

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Checking environment status...
llm_steering exists! Synchronizing any missing packages ...
Jupyter detected...
2 channel Terms of Service accepted

EnvironmentFileNotFound: '/home1/kelidari/environment.yml' file not found



In [3]:
# Install project + wandb in the conda env
!conda run -n llm_steering pip install -e . wandb 2>&1 | tail -5
print("✅ Project + wandb installed in conda env")

ERROR: file:///home1/kelidari does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
ERROR conda.cli.main_run:execute(142): `conda run pip install -e . wandb` failed. (See above for error)
Obtaining file:///home1/kelidari
✅ Project + wandb installed in conda env


## 2 · Configuration

In [4]:
import os

# ── Configurable parameters ──────────────────────────────────────────────────
MODEL_NAME     = "allenai/OLMoE-1B-7B-0125-Instruct"
N_TOKENS       = None        # middle-frequency tokens to select (None = all)
CHUNK_SIZE     = 2000           # tokens per saved chunk file
MAX_EXAMPLES   = None           # set to e.g. 500 for a quick debug run
DEVICE         = "cuda"

# Scratch directories
SCRATCH = os.environ.get("SCRATCH_DIR", os.path.expanduser("/scratch1/kelidari"))
OUTPUT_DIR     = os.path.join(SCRATCH, "dataset_3d", "output")
CHECKPOINT_DIR = os.path.join(SCRATCH, "dataset_3d", "checkpoints")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Model          = {MODEL_NAME}")
print(f"Output dir     = {OUTPUT_DIR}")
print(f"Checkpoint dir = {CHECKPOINT_DIR}")
print(f"N tokens       = {N_TOKENS}")
print(f"Chunk size     = {CHUNK_SIZE}")
print(f"Max examples   = {MAX_EXAMPLES or 'all (~87K)'}")

Model          = allenai/OLMoE-1B-7B-0125-Instruct
Output dir     = /home1/kelidari/scratch1/kelidari/dataset_3d/output
Checkpoint dir = /home1/kelidari/scratch1/kelidari/dataset_3d/checkpoints
N tokens       = None
Chunk size     = 2000
Max examples   = all (~87K)


In [5]:
L, K, E = 16, 8, 64    # OLMoE config
D = 2048                # hidden dim
N = N_TOKENS or 50000

h_ram  = N * L * D * 4 / 1e9
a_ram  = L * E * N * 4 / 1e9
x_disk = N * L * E * D * 4 / 1e9
y_disk = N * L * E * 4 / 1e9

print(f"Model config: L={L}, K={K}, E={E}, D={D}, N≈{N}")
print(f"\nRAM estimate (accumulators):")
print(f"  H_sum:  {h_ram:.2f} GB  (N, L, D)")
print(f"  A1/A2:  {a_ram:.2f} GB each  (L, E, N)")
print(f"  Total:  {h_ram + 2*a_ram:.2f} GB")
print(f"\nDisk estimate (saved dataset):")
print(f"  X:      {x_disk:.2f} GB  (N, L, E, D)")
print(f"  Y:      {y_disk:.2f} GB  (N, L, E, 1)")
print(f"  Total:  {x_disk + y_disk:.2f} GB")

Model config: L=16, K=8, E=64, D=2048, N≈50000

RAM estimate (accumulators):
  H_sum:  6.55 GB  (N, L, D)
  A1/A2:  0.20 GB each  (L, E, N)
  Total:  6.96 GB

Disk estimate (saved dataset):
  X:      419.43 GB  (N, L, E, D)
  Y:      0.20 GB  (N, L, E, 1)
  Total:  419.64 GB


## 3 · Run Dataset Generation

In [22]:
!conda run -n llm_steering python -u -m src.dataset_3d.generate \
    --model "{MODEL}" \
    --output-dir "{OUTPUT_DIR}" \
    --checkpoint-dir "{CKPT_DIR}" \
    --chunk-size {CHUNK_SIZE} \
    --device {DEVICE}

^C

CondaError: KeyboardInterrupt



## 4 · Verification

In [ ]:
!conda run -n llm_steering python -u -c "
import os, glob, torch
d = '{OUTPUT_DIR}'
meta = torch.load(os.path.join(d, 'metadata.pt'), map_location='cpu', weights_only=False)
chunks = sorted(glob.glob(os.path.join(d, 'chunk_*.pt')))
first = torch.load(chunks[0], map_location='cpu', weights_only=True)
X, Y = first['X'], first['Y']
print(f'N={meta[chr(34)+"n_total"+chr(34)]}, L={meta[chr(34)+"layers"+chr(34)]}, E={meta[chr(34)+"experts"+chr(34)]}, D={meta[chr(34)+"hidden_dim"+chr(34)]}')
print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')
assert X.dim() == 4
total = sum(os.path.getsize(f) for f in chunks)
print(f'Chunks: {len(chunks)}, Total: {total/1e9:.2f} GB')
print('All checks passed!')
"